# Lab 17 Solution Notebook
## Hyperparameter Tuning of an Artificial Neural Network for Cybersecurity Classification

This notebook uses a **Cybersecurity dataset** with **12,000 records**.
The goal is to classify each network connection as:
- **0 = normal traffic**
- **1 = suspicious / attack traffic**

The notebook is written as a guided lab.

### Dataset note
Dataset is designed for the following concepts:
- preprocessing mixed data (categorical + numerical)
- train/test splitting
- ANN baseline modeling
- hyperparameter tuning
- evaluation with classification metrics



In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

## Task 1
**Load the dataset and inspect its basic structure.**

Do the following:
1. Read the CSV file.
2. Print the shape of the dataset.
3. Display the first five rows.
4. Display the data types of all columns.

In [3]:
file_path = 'Lab17-cyber_ann_dataset.csv'
df = pd.read_csv(file_path)

print('Dataset shape:', df.shape)
display(df.head())
print('\nData types:')
print(df.dtypes)

Dataset shape: (12000, 25)


,duration_sec,protocol_type,service,flag,src_bytes,dst_bytes,src_packets,dst_packets,wrong_fragment,urgent,...,srv_count,serror_rate,rerror_rate,same_srv_rate,diff_srv_rate,dst_host_count,dst_host_srv_count,avg_packet_size,connection_rate,label
0,3.74,udp,dns,REJ,339.80,31.36,43,5,0,0,...,27,0.7976,0.5151,0.2867,0.4958,181,43,7.73,21.1044,1
1,47.25,tcp,dns,SF,1127.23,1240.91,9,23,0,0,...,9,0.0237,0.0980,0.8261,0.0923,69,44,74.00,0.2902,0
2,28.49,tcp,ssh,S0,342.36,257.92,33,10,0,0,...,93,0.5721,0.3816,0.6683,0.0670,171,96,13.96,3.5264,1
3,30.14,tcp,https,REJ,449.83,271.82,90,7,0,0,...,138,0.8780,0.2074,0.9446,0.0897,218,174,7.44,5.0415,1
4,22.83,tcp,http,SF,999.83,1205.88,9,13,0,0,...,9,0.2021,0.0196,0.5928,0.3088,63,79,100.26,0.6295,0



Data types:
duration_sec          float64
protocol_type          object
service                object
flag                   object
src_bytes             float64
dst_bytes             float64
src_packets             int64
dst_packets             int64
wrong_fragment          int64
urgent                  int64
failed_logins           int64
logged_in               int64
root_shell              int64
num_compromised         int64
count                   int64
srv_count               int64
serror_rate           float64
rerror_rate           float64
same_srv_rate         float64
diff_srv_rate         float64
dst_host_count          int64
dst_host_srv_count      int64
avg_packet_size       float64
connection_rate       float64
label                   int64
dtype: object


## Task 2
**Inspect the target variable and check data quality.**

Do the following:
1. Show the class distribution of the target variable.
2. Check whether any missing values exist.
3. Produce basic descriptive statistics.

In [ ]:
print('Class distribution:')
print(df['label'].value_counts())
print('
Class percentages:')
print(df['label'].value_counts(normalize=True).round(4))

print('
Missing values per column:')
print(df.isnull().sum())

print('
Descriptive statistics:')
display(df.describe(include='all').T)

## Task 3
**Prepare the input features and target.**

Do the following:
1. Separate the feature matrix `X` and target vector `y`.
2. Identify the categorical columns.
3. Identify the numerical columns.

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

categorical_features = ['protocol_type', 'service', 'flag']
numerical_features = [col for col in X.columns if col not in categorical_features]

print('Categorical features:', categorical_features)
print('
Numerical features:', numerical_features)

## Task 4
**Split the dataset and build a preprocessing pipeline.**

Do the following:
1. Create training and test sets.
2. One-hot encode the categorical variables.
3. Standardize the numerical variables.
4. Combine both preprocessing steps using `ColumnTransformer`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ]
)

print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)

## Task 5
**Train a baseline ANN model.**

Use a simple baseline configuration:
- one hidden layer with 32 neurons
- ReLU activation
- Adam optimizer
- small L2 regularization (`alpha`)
- early stopping enabled

Then fit the model on the training set.

In [ ]:
baseline_model = Pipeline([
    ('preprocessor', preprocessor),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(32,),
        activation='relu',
        solver='adam',
        alpha=0.0005,
        batch_size=128,
        learning_rate_init=0.001,
        max_iter=200,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=10,
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)
print('Baseline model trained successfully.')

## Task 6
**Evaluate the baseline ANN.**

Compute the following metrics on the test set:
1. Accuracy
2. Precision
3. Recall
4. F1-score
5. Confusion matrix
6. Full classification report

In [ ]:
y_pred = baseline_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1-score : {f1:.4f}')
print('
Confusion matrix:')
print(cm)
print('
Classification report:')
print(classification_report(y_test, y_pred))

## Task 7
**Perform a small manual hyperparameter tuning experiment.**

Try several ANN settings and compare their test performance.

Tune the following hyperparameters:
- hidden layer size
- activation function
- L2 regularization (`alpha`)
- learning rate

Store the results in a table and rank them by F1-score.

In [ ]:
configs = [
    {'hidden_layer_sizes': (16,), 'activation': 'relu', 'alpha': 0.0001, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (32,), 'activation': 'relu', 'alpha': 0.0005, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (64,), 'activation': 'relu', 'alpha': 0.0010, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (32, 16), 'activation': 'relu', 'alpha': 0.0005, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (64, 32), 'activation': 'relu', 'alpha': 0.0010, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (32,), 'activation': 'tanh', 'alpha': 0.0005, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (64,), 'activation': 'tanh', 'alpha': 0.0010, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (32, 16), 'activation': 'tanh', 'alpha': 0.0005, 'learning_rate_init': 0.005},
]

results = []
for cfg in configs:
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=cfg['hidden_layer_sizes'],
            activation=cfg['activation'],
            solver='adam',
            alpha=cfg['alpha'],
            batch_size=128,
            learning_rate_init=cfg['learning_rate_init'],
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.10,
            n_iter_no_change=10,
            random_state=42
        ))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'hidden_layer_sizes': str(cfg['hidden_layer_sizes']),
        'activation': cfg['activation'],
        'alpha': cfg['alpha'],
        'learning_rate_init': cfg['learning_rate_init'],
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred),
        'f1_score': f1_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values(by='f1_score', ascending=False).reset_index(drop=True)
display(results_df)

## Task 8
**Visualize the tuning results.**

Create a simple bar chart of the F1-score for each tested configuration.
This helps show how hyperparameter choices can change model performance.

In [ ]:
plot_df = results_df.copy()
plot_df['config'] = plot_df['hidden_layer_sizes'] + ' | ' + plot_df['activation']

plt.figure(figsize=(12, 5))
plt.bar(plot_df['config'], plot_df['f1_score'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('F1-score')
plt.title('Manual ANN hyperparameter tuning results')
plt.tight_layout()
plt.show()

## Task 9
**Train the final ANN using the best hyperparameter setting found above.**

Do the following:
1. Extract the best row from the tuning table.
2. Build a final ANN with those settings.
3. Train it on the training set.
4. Evaluate it again on the test set.

In [ ]:
best_row = results_df.iloc[0]
print('Best configuration found:')
display(best_row.to_frame())

best_hidden = eval(best_row['hidden_layer_sizes'])
best_activation = best_row['activation']
best_alpha = float(best_row['alpha'])
best_lr = float(best_row['learning_rate_init'])

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=best_hidden,
        activation=best_activation,
        solver='adam',
        alpha=best_alpha,
        batch_size=128,
        learning_rate_init=best_lr,
        max_iter=220,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=10,
        random_state=42
    ))
])

final_model.fit(X_train, y_train)
final_pred = final_model.predict(X_test)

print('Final model metrics:')
print(f'Accuracy : {accuracy_score(y_test, final_pred):.4f}')
print(f'Precision: {precision_score(y_test, final_pred):.4f}')
print(f'Recall   : {recall_score(y_test, final_pred):.4f}')
print(f'F1-score : {f1_score(y_test, final_pred):.4f}')
print('
Confusion matrix:')
print(confusion_matrix(y_test, final_pred))

## Task 10
**Reflection questions**

Answer the following in your own words:
1. Which hyperparameter had the biggest effect on the model in this lab?
2. Why is feature scaling important for ANN training?
3. Why should the test set not be used repeatedly during tuning?
4. Why can a larger network sometimes perform worse than a smaller one?
5. What are the limitations of using a synthetic dataset?